# Handling multiple sequences (PyTorch)

Install the Transformers, Datasets, and Evaluate libraries to run this notebook.

In [1]:
!pip install datasets evaluate transformers[sentencepiece]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.7 MB/s eta 0:00:00


In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForSequenceClassification.from_pretrained(checkpoint)

sequence = "I've been waiting for a HuggingFace course my whole life."

tokens = tokenizer.tokenize(sequence)
ids = tokenizer.convert_tokens_to_ids(tokens)
input_ids = torch.tensor(ids)
# This line will fail.
model(input_ids)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

RuntimeError: The size of tensor a (14) must match the size of tensor b (512) at non-singleton dimension 1

In [3]:
tokenized_inputs = tokenizer(sequence, return_tensors="pt")
print(tokenized_inputs["input_ids"])

tensor([[  101,  1045,  1005,  2310,  2042,  3403,  2005,  1037, 17662, 12172,
          2607,  2026,  2878,  2166,  1012,   102]])


In [4]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForSequenceClassification.from_pretrained(checkpoint)

sequence = "I've been waiting for a HuggingFace course my whole life."

tokens = tokenizer.tokenize(sequence)
ids = tokenizer.convert_tokens_to_ids(tokens)

input_ids = torch.tensor([ids])
print("Input IDs:", input_ids)

output = model(input_ids)
print("Logits:", output.logits)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Input IDs: tensor([[ 1045,  1005,  2310,  2042,  3403,  2005,  1037, 17662, 12172,  2607,
          2026,  2878,  2166,  1012]])
Logits: tensor([[-2.7276,  2.8789]], grad_fn=<AddmmBackward0>)


In [5]:
batched_ids = [
    [200, 200, 200],
    [200, 200]
]

In [6]:
padding_id = 100

batched_ids = [
    [200, 200, 200],
    [200, 200, padding_id],
]

In [7]:
model = AutoModelForSequenceClassification.from_pretrained(checkpoint)

sequence1_ids = [[200, 200, 200]]
sequence2_ids = [[200, 200]]
batched_ids = [
    [200, 200, 200],
    [200, 200, tokenizer.pad_token_id],
]

print(model(torch.tensor(sequence1_ids)).logits)
print(model(torch.tensor(sequence2_ids)).logits)
print(model(torch.tensor(batched_ids)).logits)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tensor([[ 1.5694, -1.3895]], grad_fn=<AddmmBackward0>)
tensor([[ 0.5803, -0.4125]], grad_fn=<AddmmBackward0>)
tensor([[ 1.5694, -1.3895],
        [ 1.3374, -1.2163]], grad_fn=<AddmmBackward0>)


In [11]:
batched_ids = [
    [200, 200, 200],
    [200, 200, tokenizer.pad_token_id],
]

attention_mask = [
    [1, 1, 1],
    [1, 1, 0],
]

outputs = model(torch.tensor(batched_ids), attention_mask=torch.tensor(attention_mask))
print(outputs.logits)

tensor([[ 1.5694, -1.3895],
        [ 0.5803, -0.4125]], grad_fn=<AddmmBackward0>)


In [12]:
#sequence = sequence[:max_sequence_length]

#Writing a new code as above max sequence length is not defined

max_sequence_length = tokenizer.model_max_length
sequence = sequence[:max_sequence_length]
print("Max allowed length:", max_sequence_length)
print("Truncated sequence:", sequence)

Max allowed length: 512
Truncated sequence: I've been waiting for a HuggingFace course my whole life.


In [13]:
# Compare how different tokenizers split the same sentence
from transformers import AutoTokenizer

sentence = "I'm studying Transformer-based models at HuggingFace!"

for model_name in ["bert-base-cased", "gpt2", "roberta-base"]:
    tok = AutoTokenizer.from_pretrained(model_name)
    tokens = tok.tokenize(sentence)
    print(f"{model_name:35s} → {tokens}")
    print(f"{'':35s}   Total tokens: {len(tokens)}\n")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

bert-base-cased                     → ['I', "'", 'm', 'studying', 'Trans', '##former', '-', 'based', 'models', 'at', 'Hu', '##gging', '##F', '##ace', '!']
                                      Total tokens: 15



...

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

roberta-base                        → ['I', "'m", 'Ġstudying', 'ĠTrans', 'former', '-', 'based', 'Ġmodels', 'Ġat', 'ĠHug', 'ging', 'Face', '!']
                                      Total tokens: 13



In [14]:
# See what special tokens each model adds ([CLS], [SEP], etc.)
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")
sentence = "Using a Transformer network is simple"

# Without special tokens
without = tokenizer.tokenize(sentence)
print("Without special tokens:", without)

# With special tokens
with_special = tokenizer.encode(sentence)
print("With special tokens (IDs):", with_special)

# Decode to see what was added
print("Decoded:", tokenizer.decode(with_special))

Without special tokens: ['Using', 'a', 'Trans', '##former', 'network', 'is', 'simple']
With special tokens (IDs): [101, 7993, 170, 13809, 23763, 2443, 1110, 3014, 102]
Decoded: [CLS] Using a Transformer network is simple [SEP]


In [15]:
# Tokenize two sentences together — used in Q&A and NLI tasks
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

question = "Where do I work?"
context  = "My name is Sylvain and I work at Hugging Face in Brooklyn."

encoded_pair = tokenizer(question, context)
print("Input IDs     :", encoded_pair["input_ids"])
print("Token type IDs:", encoded_pair["token_type_ids"])
# 0 = first sentence (question), 1 = second sentence (context)

decoded = tokenizer.decode(encoded_pair["input_ids"])
print("Decoded       :", decoded)

Input IDs     : [101, 2777, 1202, 146, 1250, 136, 102, 1422, 1271, 1110, 156, 7777, 2497, 1394, 1105, 146, 1250, 1120, 20164, 10932, 10289, 1107, 6010, 119, 102]
Token type IDs: [0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
Decoded       : [CLS] Where do I work? [SEP] My name is Sylvain and I work at Hugging Face in Brooklyn. [SEP]


In [16]:
# Pad short sentences and truncate long ones in a batch
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

sentences = [
    "Hi!",
    "I have been waiting for this course my whole life.",
    "Transformers are incredibly powerful models used across many NLP tasks.",
]

encoded = tokenizer(
    sentences,
    padding=True,
    truncation=True,
    max_length=20,
    return_tensors="pt"
)

print("Input IDs shape  :", encoded["input_ids"].shape)
print("Attention mask   :\n", encoded["attention_mask"])
# 1 = real token, 0 = padding token

Input IDs shape  : torch.Size([3, 14])
Attention mask   :
 tensor([[1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])


In [17]:
# Explore the tokenizer vocabulary
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

vocab = tokenizer.get_vocab()
print("Total vocabulary size:", len(vocab))

# Look up specific words
words = ["Hello", "Transformer", "##ing", "[CLS]", "[SEP]", "[PAD]", "[UNK]"]
for word in words:
    print(f"  '{word}' → ID: {vocab.get(word, 'Not found')}")

Total vocabulary size: 28996
  'Hello' → ID: 8667
  'Transformer' → ID: Not found
  '##ing' → ID: 1158
  '[CLS]' → ID: 101
  '[SEP]' → ID: 102
  '[PAD]' → ID: 0
  '[UNK]' → ID: 100


In [18]:
# See how tokenizer handles rare, made-up, or technical words
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

rare_words = [
    "Transformerization",   # Made-up word
    "COVID-19",             # Technical term
    "HuggingFace",          # Compound proper noun
    "supercalifragilistic", # Long rare word
    "chatGPT",              # Modern term
]

for word in rare_words:
    tokens = tokenizer.tokenize(word)
    print(f"  '{word}' → {tokens}")
# Notice how unknown words get split into subword pieces!

  'Transformerization' → ['Trans', '##former', '##ization']
  'COVID-19' → ['CO', '##VI', '##D', '-', '19']
  'HuggingFace' → ['Hu', '##gging', '##F', '##ace']
  'supercalifragilistic' → ['super', '##cal', '##if', '##rag', '##ilis', '##tic']
  'chatGPT' → ['chat', '##GP', '##T']
